# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print top-level metadata
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"Version: {getattr(metadata, 'version', '')}")
print(f"License: {getattr(metadata, 'license', '')}")

print("\nAuthors (@id):")
if hasattr(metadata, 'author'):
    if isinstance(metadata.author, list):
        for author in metadata.author:
            if isinstance(author, dict) and '@id' in author:
                print(f"- {author['@id']}")
            elif isinstance(author, str):
                print(f"- {author}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Here we inspect all available `RecordSet` entries in the metadata. Each RecordSet (usually a table or resource for records) will be listed with its `@id` and name, along with available Fields.

> Note: All identifiers in this notebook refer to their Croissant schema `@id`.

In [ ]:
# List all record sets and their fields
print("Available RecordSets and their fields:\n")

# mlcroissant exposes metadata.record_sets (list of RecordSet objects)
record_sets = getattr(metadata, 'record_sets', [])

record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    record_set_ids.append(rs.id)
    print(f"  Name: {getattr(rs, 'name', '')}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print("  Fields:")
    # Each RecordSet has .fields (list of Field objects)
    for fld in getattr(rs, 'fields', []):
        print(f"    - Field @id: {fld.id}, Name: {getattr(fld, 'name', '')}, DataType: {getattr(fld, 'data_type', '')}")
    print()

print(f"Found {len(record_set_ids)} record sets.")
if len(record_set_ids) == 0:
    print("⚠️ No record sets found. Check that your dataset conforms to Croissant with RecordSets.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll extract the data for all available record sets, referencing each RecordSet by its `@id`.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}
for recset_id in record_set_ids:
    print(f"Loading records for RecordSet {recset_id} ...")
    # Each record is a dict with field @id keys
    records = list(dataset.records(record_set=recset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Loaded DataFrame with columns: {df.columns.tolist()}")
        print(df.head(), '\n')
    else:
        print("No records found for this RecordSet.\n")
# For demonstration, pick the first RecordSet if available
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's choose a numeric field from one of the loaded RecordSets (by `@id`). If the dataset does not provide a numeric field, update the field name to reflect the schema.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Pick a numeric field for demonstration. You may want to adjust this based on your schema.
if main_record_set_id is not None and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]

    # Try to automatically select a numeric field
    numeric_field = None
    for col in df.columns:
        if np.issubdtype(df[col].dropna().__class__, np.number):
            numeric_field = col
            break
    if not numeric_field:
        # Try by name guess
        candidates = [c for c in df.columns if any(keyword in c.lower() for keyword in ["amount", "quantit", "mw", "weight", "count", "abundance", "coverage", "peptide", "value"])]
        if candidates:
            numeric_field = candidates[0]
        else:
            numeric_field = df.columns[0]

    print(f"Selected numeric field for analysis: {numeric_field}")

    # Convert to numeric if possible
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    # Normalization (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping by a likely categorical attribute (e.g. 'accession', 'sample', etc.)
    group_field = None
    group_candidates = [c for c in df.columns if any(k in c.lower() for k in ["accession", "sample", "group", "type", "class"])]
    if group_candidates:
        group_field = group_candidates[0]

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
        print(f"\nGrouped data by '{group_field}':")
        print(grouped_df.head())
else:
    print("No data loaded in DataFrame for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and main_record_set_id in dataframes and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if present
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library via its Croissant schema.
- We examined the available record sets, fields, and loaded records into Pandas DataFrames for flexible analysis.
- We demonstrated basic exploratory data analysis, normalization, grouping, and visualization of a chosen numeric field (e.g., protein abundance or molecular weight).

**Next Steps:**
- Explore other fields or record sets for domain-specific insights.
- Apply analysis techniques relevant to your research question, such as differential protein expression, clustering, or integration with other omics datasets.

For more advanced usage of Croissant datasets, refer to the [`mlcroissant` documentation](https://mlcommons.github.io/croissant/python-api.html).